# notebook to process ParseBio data

In [ ]:
pip list # (scvi-r docker kernel is used docker pull alexanrna/scanpy-r:1.10-v6 )
# run label transfer notebook first/at the same time

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np 
from scipy import stats
import glob 
import os
import re
from matplotlib import pyplot as plt
import datetime
import anndata as ad
from scipy import sparse
from anndata import AnnData
from scipy.sparse import csr_matrix, issparse
from collections import OrderedDict

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)


# QC and processing

In [ ]:
mat_path = '/PB-001/PB-001-comb/all-sample/DGE_filtered/' # change path as necessary, to parsebio results

In [ ]:
# The DGE_filtered folder contains the expression matrix, genes, and files
adata = sc.read_mtx(mat_path + 'count_matrix.mtx')


# reading in gene and cell data
gene_data = pd.read_csv(mat_path + 'all_genes.csv')
cell_meta = pd.read_csv(mat_path + 'cell_metadata.csv')

# find genes with nan values and filter
gene_data = gene_data[gene_data.gene_name.notnull()]
notNa = gene_data.index
notNa = notNa.to_list()

# remove genes with nan values and assign gene names
adata = adata[:,notNa]
adata.var = gene_data
adata.var.set_index('gene_name', inplace=True)
adata.var.index.name = None
adata.var_names_make_unique()

# add cell meta data to anndata object
adata.obs = cell_meta
adata.obs.set_index('bc_wells', inplace=True)
adata.obs.index.name = None
adata.obs_names_make_unique()

sc.pp.filter_cells(adata, min_counts=100)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata.shape

In [ ]:
adata.layers["counts"] =  adata.X

In [ ]:
adata.obs

In [ ]:
adata

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, )

In [ ]:
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)
adata

In [ ]:
p1 = sns.displot(adata.obs["total_counts"], bins=100, kde=False)
# sc.pl.violin(adata, 'total_counts')
p2 = sc.pl.violin(adata, "pct_counts_mt")
p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
p4 = sc.pl.violin(adata, "pct_counts_ribo")
p4 = sc.pl.violin(adata, "pct_counts_hb")

In [ ]:
metric = "pct_counts_mt"

M = adata.obs[metric]
fig, ax = plt.subplots()
ax.hist(adata.obs[metric], bins=100)
ax.set_xlabel(metric)
ax.set_ylabel('Frequency')
ax.legend(['Data'])
plt.show()

In [ ]:
p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False)

## Detect MAD outliers

In [ ]:
def is_outlier(adata, metric: str, nmads: int):
    """
    calculate median absolute deviation (MAD) to get find outliers
    """
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * stats.median_abs_deviation(M)) | (
        np.median(M) + nmads * stats.median_abs_deviation(M) < M
    )
    return outlier

In [ ]:
adata.obs["outlier"] = (
        is_outlier(adata, "log1p_total_counts", 5)
        | is_outlier(adata, "log1p_n_genes_by_counts", 5)
        | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
        )
print(adata.obs.outlier.value_counts())

In [ ]:
adata = adata[(~adata.obs.outlier)].copy()  # remove outlier cells

In [ ]:
# mitochondria
adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 5) | (
    adata.obs["pct_counts_mt"] > 12.5
    )
print(adata.obs.mt_outlier.value_counts())

In [ ]:
adata = adata[(~adata.obs.mt_outlier)].copy()

# Normalisation

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data:
sc.pp.log1p(adata)
adata.layers["log1p_norm"] = adata.X.copy()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
axes[0].set_title("Total counts")
p2 = sns.histplot(adata.layers["log1p_norm"].sum(1), bins=100, kde=False, ax=axes[1])
axes[1].set_title("Shifted logarithm")
plt.show()

# Feature selection 

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000)

In [ ]:
sc.pl.highly_variable_genes(adata)

# Dimensionality reduction

In [ ]:
sc.pp.pca(adata)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
sc.pl.pca_scatter(adata, color="total_counts")

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.tl.tsne(adata)

In [ ]:
res = 1.2
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))
res = 1.5
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))
res = 1.8
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))
res = 2.0
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))
res = 2.5
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "pct_counts_mt"]
)

In [ ]:
sc.pl.tsne(
    adata,
    color=["total_counts", "pct_counts_mt"] 
)

In [ ]:
adata

In [ ]:
for leiden in ["leiden_2.0" ]:
    
    sc.pl.umap(
        adata,
        color=leiden
        )

In [ ]:
sc.pl.umap(
    adata,
    color=[ "sample"]
)

In [ ]:
sc.pl.umap(
    adata,
    color=[ "MALAT1"]
)

# Metadata

## Region annotation

In [ ]:
adata.obs["region"] = "Cingulate Gyrus"

In [ ]:
adata.obs

# Vireo assignment <br>
## Donor assignments from Vireo/CellSNP

In [ ]:
vireo_files = glob.glob('vireo_PB-001/*')

In [ ]:
vireo_files[:5]

In [ ]:
def append_name_if_donor(row):
    if re.search(r'donor', row['donor_id'], re.IGNORECASE):
        return name + '_' + row['donor_id']
    return row['donor_id']

In [ ]:
df_merged = pd.DataFrame()
for file in vireo_files:
    print(file)
    name=file.replace('vireo/','')
    name=name.split('.')[0]
    print('\t' + name)
    
    df=pd.read_csv(file, sep='\t')
    df['donor_id'] = df.apply(append_name_if_donor, axis=1)
    df['full_barcode'] = df['cell']
    df['full_barcode']
    df.index = df['full_barcode']
    df = df[df.index.isin(adata.obs.index)]
    print(f'\t{str(len(df))}')
    df_merged = pd.concat([df_merged, df])

In [ ]:
df_merged['donor_id'].head()

In [ ]:
df_merged['donor_id'].value_counts()

In [ ]:
def print_full(x):
    pd.set_option('display.max_rows', len(x))
    print(x)
    pd.reset_option('display.max_rows')

In [ ]:
print_full(df_merged['donor_id'].value_counts())

In [ ]:
adata.obs['vireo_assignment'] = df_merged['donor_id']

In [ ]:
adata.obs['vireo_assignment'].head()

In [ ]:
donor_vireo = {"63B": "ASA_063"} # rename donor

adata.obs['vireo_assignment'] = adata.obs['sample'].map(donor_vireo).combine_first(adata.obs['vireo_assignment'])

In [ ]:
adata.obs['vireo_assignment'].value_counts()

In [ ]:
sc.pl.umap(adata, color = 'vireo_assignment') # batch effect check

In [ ]:
sc.pl.umap(adata, color = 'vireo_assignment', groups ='unassigned')

In [ ]:
sc.pl.umap(adata, color = 'vireo_assignment', groups ='doublet') 

In [ ]:
sc.pl.umap(adata, color = 'vireo_assignment', groups ='ASA_105') 

In [ ]:
adata

# Cell type annnotation

## Marker genes 

In [ ]:
marker_genes = {
    "neurons": ["SYT1"],
    "OLG": ["MAG", "MOBP"],
    "OPC": [
        "PDGFRA"],
    "Ast":["AQP4", "GFAP"],
    "Micro":["CD74","RUNX1"],
    "Endo":["CLDN5"]}

In [ ]:
SN = {
    "neurons",
    "OLG",
    "OPC",
    "Ast",
    "Micro",
    "Endo"}

In [ ]:
marker_genes_in_data = dict()
for ct, markers in marker_genes.items():
    markers_found = list()
    for marker in markers:
        if marker in adata.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

In [ ]:
for ct in SN:
    print(f"{ct.upper()}:")  
    sc.pl.tsne(
        adata,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax="p99",  
        sort_order=False, 
        frameon=False,
        cmap="Reds", 
    )
    print("\n\n\n")  

In [ ]:
for ct in SN:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adata,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax="p99",  
        sort_order=False,  
        frameon=False,
        cmap="Reds",  
    )
    print("\n\n\n")  

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color="leiden_1.8", legend_loc="on data")
sc.pl.tsne(adata, color="leiden_1.8", legend_loc="on data")

## Add library info

In [ ]:
adata.obs['library'] = adata.obs.index.str.split('__').str[-1]

In [ ]:
adata.obs['library'] 

In [ ]:
sc.pl.umap(adata, color=["library","sample","vireo_assignment"])

# Import predicted cell types from scANVI

In [ ]:
adata

In [ ]:
# Load the CSV file into a dictionary
df = pd.read_csv('./scANVI/20250209_PB-001_celltype_from_scANVI.csv')
data_dict = df.set_index('index')['column_name'].to_dict()


In [ ]:
# Map the values from the dictionary to a new column
adata.obs['subclass_scANVI'] = adata.obs.index.map(data_dict)

In [ ]:
sc.pl.umap(
    adata,
    color="subclass_scANVI", # Add any other columns here
    frameon=False,
)

In [ ]:
# Load the CSV file into a dictionary
df = pd.read_csv('./scANVI/20250209_PB-001_subclusters_from_scANVI.csv')
data_dict = df.set_index('index')['column_name'].to_dict()


In [ ]:
# Map the values from the dictionary to a new column
adata.obs['cell_type_scANVI'] = adata.obs.index.map(data_dict)

In [ ]:
sc.pl.umap(
    adata,
    color="cell_type_scANVI", # Add any other columns here
    frameon=False,
)

In [ ]:
sc.pl.umap(adata, color="region")

# Visualisations + L4 annotation

In [ ]:
from matplotlib.patches import Patch

In [ ]:
value_counts = adata.obs['vireo_assignment'].value_counts()
sns.set_style("whitegrid")

# Create a bar plot
plt.figure(figsize=(8, 5))
value_counts.plot(kind='bar')

# Customize x-axis ticks
ticks = range(len(value_counts))
labels = value_counts.index

plt.xlabel('Vireo assignment')
plt.ylabel('Number of cells')
plt.xticks(ticks, labels, fontsize=9, rotation=45, ha="right")
plt.grid(False)
plt.show()

In [ ]:
adata.obs['vireo_assignment'].value_counts()

In [ ]:
adata.obs['vireo_assignment'].value_counts().sum()

In [ ]:
sc.pl.umap(adata, color = 'leiden_2.0', groups = '28') 
sc.pl.umap(adata, color = 'vireo_assignment', groups = 'doublet') 

In [ ]:
sc.pl.umap(adata, color = ["RORB","CUX2", "CTNNB1","GRIN3A"])

# Doublet removal

In [ ]:
# what is fraction of doublets per cluster

# init lists
list_clusters = np.unique(adata.obs['leiden_2.5'])
list_size = []
list_frac_doublets = []

# loop over clusters
for clus in list_clusters:
    clus_adata = adata[adata.obs['leiden_2.5'] == clus]
    
    size = clus_adata.shape[0]
    n_doublets = sum( clus_adata.obs['vireo_assignment'] == 'doublet')
    
    list_frac_doublets.append( n_doublets / size )
    list_size.append(size)
    
df_frac_doublets = pd.DataFrame(
{
    'cluster': list_clusters,
    'size': list_size,
    'frac_doublets': list_frac_doublets
}
)

In [ ]:
df_frac_doublets.sort_values('frac_doublets', ascending=False)

In [ ]:
doublet_clusters = [ clus for clus 
                    in df_frac_doublets[ df_frac_doublets['frac_doublets'] >= 0.3 ]['cluster'] ]
doublet_clusters

In [ ]:
# remove doublet clusters and doublets
adata = adata[ ~adata.obs['leiden_2.5'].isin(doublet_clusters) ].copy()
adata = adata[ adata.obs['vireo_assignment'] != 'doublet' ].copy()
adata = adata[ adata.obs['vireo_assignment'] != 'unassigned' ].copy()
adata

# recluster the data

In [ ]:
adata.X = adata.layers["counts"].copy()
print(np.unique(adata.X.data))
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data:
sc.pp.log1p(adata)
adata.layers["log1p_norm"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
sc.pp.pca(adata)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
sc.pl.pca_scatter(adata, color="total_counts")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color="cell_type_scANVI"
)

In [ ]:
with plt.rc_context({'figure.figsize': (15, 5)}):
    sc.pl.violin(adata, [ "gene_count"], groupby='cell_type_scANVI', stripplot=False, inner='box')  # use stripplot=False to remove the internal dots, inner='box' adds a boxplot inside violins


In [ ]:
with plt.rc_context({'figure.figsize': (20, 5)}):
    sc.pl.violin(adata, [ "gene_count"], groupby='vireo_assignment', stripplot=False, inner='box')  # use stripplot=False to remove the internal dots, inner='box' adds a boxplot inside violins


In [ ]:
with plt.rc_context({'figure.figsize': (20, 5)}):
    sc.pl.violin(adata, [ "log1p_total_counts_mt"], groupby='vireo_assignment', stripplot=False, inner='box')  # use stripplot=False to remove the internal dots, inner='box' adds a boxplot inside violins


In [ ]:
with plt.rc_context({'figure.figsize': (20, 5)}):
    sc.pl.violin(adata, [ "log1p_total_counts_mt"], groupby='cell_type_scANVI', stripplot=False, inner='box')  # use stripplot=False to remove the internal dots, inner='box' adds a boxplot inside violins


In [ ]:
with plt.rc_context({'figure.figsize': (20, 5)}):
    sc.pl.violin(adata, [ "MALAT1"], groupby='vireo_assignment', stripplot=False, inner='box')  # use stripplot=False to remove the internal dots, inner='box' adds a boxplot inside violins


In [ ]:
with plt.rc_context({'figure.figsize': (20, 5)}):
    sc.pl.violin(adata, [ "gene_count"], groupby='cell_type_scANVI', stripplot=False, inner='box')  # use stripplot=False to remove the internal dots, inner='box' adds a boxplot inside violins

# marker genes

In [ ]:
sc.tl.rank_genes_groups(
    adata, groupby="cell_type_scANVI", method="wilcoxon", key_added="dea_cell_type_scANVI"
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="cell_type_scANVI", standard_scale="var", n_genes=5, key="dea_cell_type_scANVI", dendrogram = True
)

In [ ]:
sc.pl.umap(adata, color=[ "SLC35F3", "SV2C", "PHACTR1", "HTR2C", "MACROD2"], frameon=False, )

## Add metadata

In [ ]:
def remove_trailing_letter(subject_id):
    if subject_id.endswith('A') or subject_id.endswith('B') or subject_id.endswith('C') or subject_id.endswith('D'):
        return subject_id[:-1]
    return subject_id

In [ ]:
# add metadta

metadata_path = '/donor_statistics/crn_metadata/SUBJECT.csv' # change path as necessary, to metadata file
metadata = pd.read_csv(metadata_path)

metadata['subject_id'] = metadata['subject_id'].apply(remove_trailing_letter)
metadata_cleaned = metadata.drop_duplicates()

columns_to_map = ['biobank_name','sex','age_at_collection','primary_diagnosis','age_at_diagnosis']
mapping_dicts = {col: metadata_cleaned.set_index('subject_id')[col].to_dict() for col in columns_to_map}


for col, mapping_dict in mapping_dicts.items():
    adata.obs[col] = adata.obs['vireo_assignment'].map(mapping_dict)

adata.obs
for c in columns_to_map:
    sc.pl.umap(
    adata,
    color=c,
)

In [ ]:
date = datetime.datetime.now().strftime('%Y%m%d')
adata.write(f'./matrices/{date}_PB-001_filtered_umap_tsne_annotated_scANVI_metadata.h5ad')

In [ ]:
import session_info

session_info.show()

In [ ]:
pip list